# -- Notebook 01: Cleaning --

**Summary of the notebook:**

* Cleans the dataset that was setup at the end of notebook 00
* Shows the downsides to simply deleting missing values 
* Imputes missing values using simple mean/median values 
* Removes irrelevant variables 
* Checks for and removes any duplicate values 
* Splits the dataset into test/train data


In [32]:
# Necessary libraries: 

import pandas as pd
import miceforest as mf 
import numpy as np 
from sklearn.model_selection import train_test_split 
%config InlineBackend.figure_format = "retina"
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from pathlib import Path


In [257]:
# Base path - edit as necessary: 
base_path = Path(r"C:/Users/abhik/OneDrive/Desktop/WISE")

In [258]:
collisions_df_path = base_path / "Data" / "Processed" / "collisions_00.csv"
collisions_df = pd.read_csv(collisions_df_path)

In [259]:
# Taking the recommendation from the warning message above: 

dtype_map = {'collision_index': str, 'collision_ref_no': str}
collisions_df = pd.read_csv(collisions_df_path, dtype = dtype_map)

**Downside to simple deletion of observations with missing values (for this dataset)** 

In [260]:
value_counts_str = (collisions_df == '-1').sum()
value_counts_num = (collisions_df == -1).sum()

# In the dataset, -1 is used to represent a missing value. 
# However, in some variables with string values, -1 has been coded as a string value, whereas in other variables with numerical/integer values, -1 has been coded as a an integer. 
# Therefore, we must consider both data types when examining how many missing values are present. 

In [261]:
# Missing values coded as strings: 
with pd.option_context('display.max_rows', None):
    print(value_counts_str.sort_values(ascending=False)) 

generic_make_model                                  278193
lsoa_of_driver                                      272994
lsoa_of_casualty                                    171016
lsoa_of_accident_location                            50287
collision_index                                          0
date                                                     0
first_road_class                                         0
local_authority_highway_current                          0
local_authority_highway                                  0
local_authority_ons_district                             0
local_authority_district                                 0
time                                                     0
day_of_week                                              0
number_of_vehicles                                       0
number_of_casualties                                     0
road_type                                                0
collision_severity                                      

In [262]:
# Missing values coded as integers/numeric: 
with pd.option_context('display.max_rows', None):
    print(value_counts_num.sort_values(ascending=False))

driver_distance_banding                             1027143
casualty_distance_banding                           1009754
local_authority_district                            1002680
enhanced_severity_collision                          554891
junction_control                                     527700
enhanced_casualty_severity                           510996
second_road_number                                   478926
engine_capacity_cc                                   275679
age_of_vehicle                                       260551
propulsion_code                                      260269
driver_imd_decile                                    231459
age_band_of_driver                                   161069
age_of_driver                                        161069
vehicle_manoeuvre_historic                           160211
special_conditions_at_site                           159537
carriageway_hazards_historic                         159519
pedestrian_crossing_human_control_histor

In [263]:
missing_cols_1 = collisions_df.loc[:, value_counts_num > 0]
missing_cols_2 = collisions_df.loc[:, value_counts_str > 0]

In [264]:
collisions_nomiss = collisions_df[(missing_cols_2 != "-1").all(axis=1)]
collisions_nomiss = collisions_nomiss[(missing_cols_1 != -1).all(axis=1)]

C:\Users\abhik\AppData\Local\Temp\ipykernel_33496\3316343178.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  collisions_nomiss = collisions_nomiss[(missing_cols_1 != -1).all(axis=1)]


In [265]:
collisions_nomiss.head(10)

,collision_index,collision_year,collision_ref_no,vehicle_reference_x,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight


In [266]:
collisions_nomiss.shape

(0, 93)

This illustrates the limitations of simply removing rows with missing values, as a substantial number of rows have at least one missing value. I thus impute the missing values for the predictors (the set of X variables). Our outcome variable is collision_severity, which has no missing values. 

**Converting time variables:**

In [ ]:
# Splitting date into separate variables: 
collisions_df["date"] = pd.to_datetime(collisions_df["date"], dayfirst=True)

collisions_df["year"] = collisions_df["date"].dt.year
collisions_df["month"] = collisions_df["date"].dt.month
collisions_df["dayofweek"] = collisions_df["date"].dt.dayofweek

In [ ]:
# Splitting time into separate variables: 
df["time"] = pd.to_datetime(df["time"])
df["hour"] = df["time"].dt.hour

**Removing missing values from the outcome variable**

In [267]:
# Making sure that there are no missing values in the outcome variable: 
collisions_df = collisions_df[collisions_df["collision_severity"] != -1]
collisions_df = collisions_df[collisions_df["collision_severity"] != "-1"]

In [268]:
collisions_df.shape
    # We see that even though we have removed only the missing values for the outcome variable, we have still lost quite a few observations (around 400,000 observations)
    # This can prove quite costly for prediction models 

(1221287, 93)

In [269]:
collisions_df.head(10)

,collision_index,collision_year,collision_ref_no,vehicle_reference_x,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,2020010280094,2020,010280094,1,1,3,2,24,5,3,...,0,0,1,3,2,E01003538,-1,0,0.0,1.0
1,2020010280094,2020,010280094,1,1,3,2,24,5,3,...,0,0,1,3,2,E01003538,-1,0,0.0,1.0
2,202031D109620,2020,31D109620,1,1,3,2,95,11,2,...,0,0,1,1,2,E01028156,-1,0,1.0,0.0
3,2020401003715,2020,401003715,1,1,3,1,39,7,3,...,0,0,1,1,2,E01017466,3,1,0.0,1.0
4,2021201086330,2021,201086330,1,1,3,1,63,9,3,...,0,0,1,2,2,E01009200,3,1,0.0,1.0
5,2021371050963,2021,371050963,1,1,3,1,51,8,3,...,0,0,1,3,2,E01030190,3,1,0.0,1.0
6,2022161255871,2022,161255871,1,1,3,2,60,9,3,...,0,0,1,1,2,E01033104,3,1,0.0,1.0
7,2023201345434,2023,201345434,1,1,3,1,72,10,3,...,0,0,1,1,2,E01009007,3,1,0.0,1.0
8,2024041434010,2024,041434010,1,1,3,1,39,7,2,...,-1,0,2,1,1,E01025105,5,1,1.0,0.0
9,2020140992628,2020,140992628,2,1,1,1,17,4,3,...,0,0,2,1,2,E01007596,3,1,0.0,1.0


In [270]:
collisions_df.tail(10)

,collision_index,collision_year,collision_ref_no,vehicle_reference_x,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
1221277,2020137CF1699,2020,137CF1699,1,1,3,1,37,7,3,...,0,0,1,1,2,E01011681,-1,0,0.128801,0.871199
1221278,2021552100103,2021,552100103,1,1,3,2,48,8,2,...,0,0,1,1,2,E01015321,-1,0,1.000000,0.000000
1221279,2022211244451,2022,211244451,1,1,3,2,7,2,2,...,0,0,1,1,2,E01029360,7,1,1.000000,0.000000
1221280,2022430195626,2022,430195626,1,1,3,2,35,6,3,...,0,0,2,3,2,E01028616,-1,0,0.026442,0.973558
1221281,2022451135038,2022,451135038,1,1,3,2,12,3,2,...,0,0,1,1,2,E01030580,7,1,1.000000,0.000000
1221282,2023421294508,2023,421294508,1,1,3,1,50,8,3,...,0,0,1,1,2,E01033141,3,1,0.000000,1.000000
1221283,2024311507759,2024,311507759,2,1,1,1,25,5,2,...,-1,0,1,1,2,E01013846,7,1,1.000000,0.000000
1221284,2024311507759,2024,311507759,2,1,1,1,25,5,2,...,-1,0,1,1,2,E01013846,7,1,1.000000,0.000000
1221285,2024010503902,2024,010503902,1,1,1,1,57,9,3,...,0,0,1,3,2,E01003808,-1,0,0.000000,1.000000
1221286,2024010503902,2024,010503902,1,1,1,1,57,9,3,...,0,0,1,3,2,E01003808,-1,0,0.000000,1.000000


**Imputing missing values**

In [271]:
# Recoding "-1" as NaN: 

collisions_df = collisions_df.replace(-1, np.nan)
collisions_df = collisions_df.replace("-1", np.nan)


In [272]:
# To avoid keeping columns with all missing values, I remove variables with only missing values

all_null = collisions_df.columns[collisions_df.isnull().all()]
collisions_df = collisions_df.drop(columns=all_null)

In [273]:
# Splitting into test/train data chronologically:

before_2024 = collisions_df["collision_year"] < 2024
after_2024 = collisions_df["collision_year"] == 2024 

train_df = collisions_df[before_2024]
test_df = collisions_df[after_2024]

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")


Training data shape: (977848, 93)
Test data shape: (243439, 93)


In [274]:
all_null_train = train_df.columns[train_df.isnull().all()]
all_null_train

Index(['casualty_distance_banding', 'driver_distance_banding'], dtype='str')

In [275]:
all_null_test = test_df.columns[test_df.isnull().all()]
all_null_test

Index(['local_authority_district'], dtype='str')

In [276]:
train_df = train_df.drop(columns=all_null_train)
train_df = train_df.reset_index(drop = True)

test_df = test_df.drop(columns=all_null_train)
test_df = test_df.reset_index(drop = True)

In [277]:
# List of numeric-coded and string-coded variables: 

numerical_features = train_df.select_dtypes(include="number").columns
string_features = train_df.select_dtypes(include=["object", "string", "category"]).columns

*Imputing train data:*

In [278]:
train_df[numerical_features] = train_df[numerical_features].astype("float32")
    # This is to cut memory usage 

# Encoding string-coded features as categories for imputation:     
for col in string_features:
    train_df[col] = train_df[col].astype("category")

In [279]:
# Removing high-cardinality columns: 

for col in string_features:
    print(col, train_df[col].nunique())
    
train_df_dropped = train_df.drop(
    columns=["lsoa_of_accident_location", "lsoa_of_casualty", "lsoa_of_driver"])

# Removing ID columns: 

train_ids = ["collision_index", "collision_ref_no"]
train_df_dropped = train_df_dropped.drop(train_ids, axis = 1, errors = "ignore")

collision_index 402548
collision_ref_no 402235
lsoa_of_casualty 34429
generic_make_model 995
lsoa_of_driver 34648
date 1461
time 1440
local_authority_ons_district 375
local_authority_highway 210
local_authority_highway_current 207
lsoa_of_accident_location 33416


In [281]:
# Combining rare categories: 

def group_rare_cats(df, column, min_count = 150):
    counts =df[column].value_counts()
    rare = counts[counts < min_count].index
    
    if df[column].dtype.name == "category":
        if "RARE_CATEGORY" not in df[column].cat.categories:
            df[column] = df[column].cat.add_categories(["RARE_CATEGORY"])
        
    df[column] = df[column].replace(rare, "RARE_CATEGORY")
    return df

cat_cols = train_df_dropped.select_dtypes(include = ["object", "category", "string"]).columns.tolist()

for col in cat_cols:
    train_df_dropped = group_rare_cats(train_df_dropped, col, min_count = 150)
    train_df_dropped[col] = train_df_dropped[col].astype("category")

In [ ]:
# Creating kernel: 
train_kernel = mf.ImputationKernel(train_df_dropped, save_all_iterations_data=False, random_state = 72)

# Running MICE algorithm for 2 iterations: 
train_kernel.mice(iterations = 1, verbose = True, min_data_in_leaf = 150)

# Completed dataset: 
train_imputed = train_kernel.complete_data()

# Adding IDs back in: 
train_imputed["collision_index"] = train_ids["collision_index"]
train_imputed["collision_ref_no"] = train_ids["collision_ref_no"]

*Imputing test data:*

In [ ]:
test_df[numerical_features] = test_df[numerical_features].astype("float32")

# Encoding string-coded features as categories for imputation:     
for col in string_features:
    test_df[col] = test_df[col].astype("category")

# Removing ID column: 
test_ids = test_df[["collision_index", "collision_ref_no"]]
test_df = test_df.drop(train_ids, axis = 1)

# Creating kernel for test data:
test_kernel = train_kernel.impute_new_data(test_df)

test_kernel.mice(3)
test_imputed = test_kernel.complete_data()

# Adding IDs back in: 
test_imputed["collision_index"] = test_ids["collision_index"]
test_imputed["collision_ref_no"] = test_ids["collision_ref_no"]

In [ ]:
# # Handling missing values for numerical features

# for col in numerical_features:
#     median = train_df[col].median()
#     train_df[col] = train_df[col].fillna(median)
#     test_df[col] = test_df[col].fillna(median)

In [ ]:
# # Handling missing values and encode categorical features

# for col in string_features:
    
#     mode_series = train_df[col].mode()
#     mode = mode_series.iloc[0] 
    
#     train_df[col] = train_df[col].fillna(mode).astype(str)
#     test_df[col] = test_df[col].fillna(mode).astype(str)
    
#     # Use standard LabelEncoder workflow to avoid indexer mismatches
#     le = LabelEncoder()
#     # Handle unseen labels in test set gracefully by creating an 'unknown' class mapping if needed
#     train_df[col] = le.fit_transform(train_df[col])
    
#     # Map unseen test labels to a default value to prevent -1 generation
#     le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
#     test_df[col] = test_df[col].apply(lambda x: le_dict.get(x, -1))

    

In [ ]:
# Checking if there are any missing values after imputation: 

print("Missing values in training set:", train_imputed.isnull().sum().sum())
print("Missing values in test set:", test_imputed.isnull().sum().sum())

Missing values in training set: 0
Missing values in test set: 0


In [ ]:
# Displaying a sample of the transformed data
print("\nTransformed training data sample:")
train_imputed.head(20)


Transformed training data sample:


,collision_index,collision_year,collision_ref_no,vehicle_reference_x,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,18242,2020,18242,1,1,3,2.0,24.0,5.0,3,...,0.0,0.0,1.0,3.0,2.0,3418,3.0,0,0.000000,1.000000
1,18242,2020,18242,1,1,3,2.0,24.0,5.0,3,...,0.0,0.0,1.0,3.0,2.0,3418,3.0,0,0.000000,1.000000
2,50118,2020,222187,1,1,3,2.0,95.0,11.0,2,...,0.0,0.0,1.0,1.0,2.0,26481,3.0,0,1.000000,0.000000
3,58224,2020,254009,1,1,3,1.0,39.0,7.0,3,...,0.0,0.0,1.0,1.0,2.0,16408,3.0,1,0.000000,1.000000
4,138546,2021,183169,1,1,3,1.0,63.0,9.0,3,...,0.0,0.0,1.0,2.0,2.0,8669,3.0,1,0.000000,1.000000
5,154624,2021,250157,1,1,3,1.0,51.0,8.0,3,...,0.0,0.0,1.0,3.0,2.0,28306,3.0,1,0.000000,1.000000
6,237796,2022,172392,1,1,3,2.0,60.0,9.0,3,...,0.0,0.0,1.0,1.0,2.0,31060,3.0,1,0.000000,1.000000
7,345924,2023,192752,1,1,3,1.0,72.0,10.0,3,...,0.0,0.0,1.0,1.0,2.0,8501,3.0,1,0.000000,1.000000
8,37336,2020,160454,2,1,1,1.0,17.0,4.0,3,...,0.0,0.0,2.0,1.0,2.0,7183,3.0,1,0.000000,1.000000
9,37336,2020,160454,2,1,1,1.0,17.0,4.0,3,...,0.0,0.0,2.0,1.0,2.0,7183,3.0,1,0.000000,1.000000


In [ ]:
Splitting train and test data into the predictors and outcome variable: 

X_train = train_imputed.drop(columns = ["collision_severity", "enhanced_severity_collision"])
y_train = train_imputed[["collision_severity", "collision_ref_no", "enhanced_severity_collision"]]

X_test = test_imputed.drop(columns = ["collision_severity", "enhanced_severity_collision"])
y_test = test_imputed[["collision_severity", "collision_ref_no", "enhanced_severity_collision"]]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (977848, 89)
X_test shape: (243439, 89)
y_train shape: (977848, 3)
y_test shape: (243439, 3)


**Exporting the data:**

In [ ]:
X_train_path = base_path / "Data" / "Processed" / "X_train.csv"
X_train.to_csv(X_train_path, index = False)

X_test_path = base_path / "Data" / "Processed" / "X_test.csv"
X_test.to_csv(X_test_path, index = False)

y_train_path = base_path / "Data" / "Processed" / "y_train.csv"
y_train.to_csv(y_train_path, index = False)

y_test_path = base_path / "Data" / "Processed" / "y_test.csv"
y_test.to_csv(y_test_path, index = False)